In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Load valuations and player metadata
# Adjust file paths if they are in 'data/' or your root folder
valuations_df = pd.read_csv('player_valuations.csv')
players_df = pd.read_csv('players.csv')

# 2. Merge on player_id to attach demographics and position
tm_df = pd.merge(
    valuations_df,
    players_df,
    on='player_id',
    how='inner',
    suffixes=('_val', '_profile')
)

# 3. Filter strictly for Goalkeepers
tm_gk = tm_df[tm_df['position'] == 'Goalkeeper'].copy()

# 4. Standardize the valuation date and derive the season year
tm_gk['date'] = pd.to_datetime(tm_gk['date'], errors='coerce')
# Valuation season roughly corresponds to the calendar year (or split by July)
tm_gk['season'] = tm_gk['date'].dt.year

print(f"Loaded {len(tm_gk)} goalkeeper valuation records.")

Loaded 72205 goalkeeper valuation records.


In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path
from thefuzz import process, fuzz
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# 1. LOAD & FILTER DATA
# ==========================================
# Load FBref Keeper stats
fbref_gk = pd.read_csv('data/fbref_raw/fbref_keeper.csv')

# Standardize seasons to integer years (e.g., '2324' -> 2023)
if fbref_gk['season'].dtype == object:
    season_map = {'2223': 2022, '2324': 2023, '2425': 2024, '2526': 2025}
    fbref_gk['season'] = fbref_gk['season'].map(season_map).fillna(fbref_gk['season'])
fbref_gk['season'] = fbref_gk['season'].astype(int)

# ==========================================
# 2. FUZZY MATCH & MERGE
# ==========================================
unique_tm_names = tm_gk['name'].unique()
fbref_names = fbref_gk['player'].unique().tolist()

print("Matching Goalkeeper names...")
name_map = {}
for name in unique_tm_names:
    match = process.extractOne(name, fbref_names, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 80:
        name_map[name] = match[0]

tm_gk['matched_fbref_name'] = tm_gk['name'].map(name_map)

# Merge on Name and Season
gk_model_data = pd.merge(
    tm_gk, fbref_gk,
    left_on=['matched_fbref_name', 'season'],
    right_on=['player', 'season'],
    how='inner'
)

# ==========================================
# 3. FEATURE ENGINEERING & CLEANING
# ==========================================
# Date math: Age and Contract remaining
date_cols = ['date', 'date_of_birth', 'contract_expiration_date']
for col in date_cols:
    if col in gk_model_data.columns:
        gk_model_data[col] = pd.to_datetime(gk_model_data[col], errors='coerce')

if 'date_of_birth' in gk_model_data.columns and 'date' in gk_model_data.columns:
    gk_model_data['age_at_val'] = (gk_model_data['date'] - gk_model_data['date_of_birth']).dt.days / 365.25
if 'contract_expiration_date' in gk_model_data.columns and 'date' in gk_model_data.columns:
    gk_model_data['contract_days_left'] = (gk_model_data['contract_expiration_date'] - gk_model_data['date']).dt.days
    gk_model_data['contract_days_left'] = gk_model_data['contract_days_left'].fillna(0)

# Drop data leaks, raw text, and raw dates
leaky_and_noise_cols = [
    'date', 'date_of_birth', 'contract_expiration_date', 'market_value_in_eur_y', 
    'highest_market_value_in_eur', 'current_club_id_x', 'current_club_id_y', 
    'player_id', 'matched_fbref_name', 'player', 'name', 'image_url', 'url', 
    'first_name', 'last_name', 'player_code', 'current_club_name_x', 
    'current_club_name_y', 'squad', 'comp', 'nation', 'country_of_citizenship',
    'country_of_birth', 'city_of_birth'
]
gk_cleaned = gk_model_data.drop(columns=[c for c in leaky_and_noise_cols if c in gk_model_data.columns])

# Fill Keeper-specific NaNs (e.g., Save% is NaN if they faced 0 shots)
# Adjust these column names if your fbref_keeper.csv uses different prefixes
save_pct_cols = [c for c in gk_cleaned.columns if '%' in c or 'Save' in c]
gk_cleaned[save_pct_cols] = gk_cleaned[save_pct_cols].fillna(0)
gk_cleaned = gk_cleaned.fillna(gk_cleaned.median(numeric_only=True))

# ==========================================
# 4. ENCODING & TRAINING
# ==========================================
# Dynamically find the correct target column (e.g., 'market_value_in_eur' or 'market_value_in_eur_val')
target_col = [c for c in gk_cleaned.columns if c.startswith('market_value_in_eur')][0]
print(f"Target variable found: {target_col}")

y = np.log1p(gk_cleaned[target_col])
# Drop ALL market value columns from features (to prevent data leaks if both _val and _profile exist)
X = gk_cleaned.drop(columns=[c for c in gk_cleaned.columns if 'market_value' in c])

# Identify categorical and numeric columns
cat_cols = ['foot', 'team', 'league', 'agent_name']
cat_cols = [c for c in cat_cols if c in X.columns]
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Ensure X only contains the columns we are explicitly handling
X = X[cat_cols + num_cols]
X[cat_cols] = X[cat_cols].fillna('Unknown')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)],
    remainder='passthrough'
)

X_train_encoded = pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())
X_test_encoded = pd.DataFrame(preprocessor.transform(X_test), columns=preprocessor.get_feature_names_out())

# Train XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05, 
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    tree_method='hist', random_state=42
)
print("Training Goalkeeper XGBoost Model...")
xgb_model.fit(X_train_encoded, y_train)

# Evaluate
y_pred = xgb_model.predict(X_test_encoded)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

print(f"\nGoalkeeper Model R²: {r2:.4f}")
print(f"Goalkeeper Model RMSE: €{rmse:,.2f}")

# Save the GK models for the Web App
model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

import joblib
joblib.dump(xgb_model, model_dir / 'gk_xgboost_model.joblib')
joblib.dump(preprocessor, model_dir / 'gk_preprocessor.joblib')
joblib.dump(cat_cols + num_cols, model_dir / 'gk_feature_order.joblib')

Matching Goalkeeper names...
Target variable found: market_value_in_eur_val


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [5]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from thefuzz import fuzz, process
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# ==============================================================================
# 1. LOAD DATA WITH AUTOMATIC PATH RESOLUTION
# ==============================================================================
val_file = 'player_valuations.csv' if os.path.exists('player_valuations.csv') else 'data/player_valuations.csv'
players_file = 'players.csv' if os.path.exists('players.csv') else 'data/players.csv'
fbref_file = 'data/fbref_raw/fbref_keeper.csv' if os.path.exists('data/fbref_raw/fbref_keeper.csv') else 'fbref_keeper.csv'

valuations_df = pd.read_csv(val_file)
players_df = pd.read_csv(players_file)
fbref_gk = pd.read_csv(fbref_file)

# Standardize FBref column names
fbref_gk.columns = [str(c).strip() for c in fbref_gk.columns]
fbref_player_col = next((c for c in fbref_gk.columns if c.lower() in ['player', 'player_name', 'name']), fbref_gk.columns[0])
fbref_season_col = next((c for c in fbref_gk.columns if 'season' in c.lower()), None)

# ==============================================================================
# 2. FILTER TRANSFERMARKT FOR GOALKEEPERS & STANDARDIZE SEASONS
# ==============================================================================
# Attach demographics and position metadata
tm_df = pd.merge(valuations_df, players_df, on='player_id', how='inner', suffixes=('_val', '_profile'))

# Filter strictly for Goalkeepers
pos_col = next((c for c in ['position', 'sub_position', 'position_profile'] if c in tm_df.columns), None)
if pos_col:
    tm_gk = tm_df[tm_df[pos_col].astype(str).str.contains('Goalkeeper|GK', case=False, na=False)].copy()
else:
    tm_gk = tm_df.copy()

# Dynamically locate the target valuation column
target_col = next((c for c in ['market_value_in_eur_val', 'market_value_in_eur_x', 'market_value_in_eur'] if c in tm_gk.columns), None)
if not target_col:
    target_col = [c for c in tm_gk.columns if 'market_value_in_eur' in c][0]
tm_gk = tm_gk[tm_gk[target_col] > 0].copy()

# Identify date and name columns
date_col = next((c for c in ['date_val', 'date', 'valuation_date'] if c in tm_gk.columns), 'date')
tm_name_col = next((c for c in ['name', 'player_name', 'name_profile', 'name_val'] if c in tm_gk.columns), 'name')

# Derive season year from date (pre-July belongs to the preceding campaign)
tm_gk[date_col] = pd.to_datetime(tm_gk[date_col], errors='coerce')
tm_gk['season_clean'] = tm_gk[date_col].apply(
    lambda d: (d.year - 1) if (pd.notna(d) and d.month < 7) else (d.year if pd.notna(d) else np.nan)
)

# Standardize FBref seasons (e.g., '2324' -> 2023, '2023-2024' -> 2023)
def standardize_season(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if '-' in s:
        return int(s.split('-')[0])
    if len(s) == 4 and s.isdigit():
        prefix = int(s[:2])
        return 1900 + prefix if prefix > 50 else 2000 + prefix
    try:
        return int(float(s))
    except:
        return np.nan

if fbref_season_col:
    fbref_gk['season_clean'] = fbref_gk[fbref_season_col].apply(standardize_season)
else:
    fbref_gk['season_clean'] = np.nan

# ==============================================================================
# 3. FUZZY MATCH & ROBUST MERGE
# ==============================================================================
unique_tm_names = tm_gk[tm_name_col].dropna().unique()
fbref_names = fbref_gk[fbref_player_col].dropna().unique().tolist()

print(f"Fuzzy matching {len(unique_tm_names)} Transfermarkt Goalkeepers...")
name_map = {}
for name in unique_tm_names:
    match = process.extractOne(name, fbref_names, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= 75:
        name_map[name] = match[0]

tm_gk['matched_fbref_name'] = tm_gk[tm_name_col].map(name_map)

# Attempt seasonal merge first
gk_merged = pd.DataFrame()
if fbref_season_col and fbref_gk['season_clean'].notna().any():
    gk_merged = pd.merge(
        tm_gk.dropna(subset=['matched_fbref_name', 'season_clean']),
        fbref_gk.dropna(subset=[fbref_player_col, 'season_clean']),
        left_on=['matched_fbref_name', 'season_clean'],
        right_on=[fbref_player_col, 'season_clean'],
        how='inner'
    )

# Fallback to closest/latest profile if season matching yields empty results
if len(gk_merged) == 0:
    print("Exact (Name, Season) yielded 0 rows. Using latest profile match fallback...")
    fbref_dedup = fbref_gk.drop_duplicates(subset=[fbref_player_col])
    gk_merged = pd.merge(
        tm_gk.dropna(subset=['matched_fbref_name']),
        fbref_dedup,
        left_on='matched_fbref_name',
        right_on=fbref_player_col,
        how='inner'
    )

print(f"Successfully matched and merged {len(gk_merged)} goalkeeper records.")

# ==============================================================================
# 4. FEATURE ENGINEERING & LEAK PREVENTION
# ==============================================================================
dob_col = next((c for c in ['date_of_birth', 'date_of_birth_profile'] if c in gk_merged.columns), None)
exp_col = next((c for c in ['contract_expiration_date', 'contract_expiration_date_profile'] if c in gk_merged.columns), None)

if dob_col and date_col in gk_merged.columns:
    gk_merged['age_at_val'] = (pd.to_datetime(gk_merged[date_col], errors='coerce') - pd.to_datetime(gk_merged[dob_col], errors='coerce')).dt.days / 365.25
if exp_col and date_col in gk_merged.columns:
    gk_merged['contract_days_left'] = (pd.to_datetime(gk_merged[exp_col], errors='coerce') - pd.to_datetime(gk_merged[date_col], errors='coerce')).dt.days
    gk_merged['contract_days_left'] = gk_merged['contract_days_left'].fillna(0)

# Unify categorical identifiers
for c in ['team', 'squad', 'current_club_name', 'current_club_name_profile']:
    if c in gk_merged.columns:
        gk_merged['team'] = gk_merged[c]
        break

for c in ['league', 'comp', 'domestic_competition_id']:
    if c in gk_merged.columns:
        gk_merged['league'] = gk_merged[c]
        break

# Isolate target variable (Log scale)
y = np.log1p(gk_merged[target_col])

# Strip IDs, URLs, and duplicate market valuation columns to prevent data leakage
leak_patterns = ['market_value', 'player_id', 'url', 'image', 'first_name', 'last_name', 
                 'player_code', 'date', 'name', 'season_clean', 'matched_fbref_name', 'country', 'city']
cols_to_drop = [c for c in gk_merged.columns if any(p in c.lower() for p in leak_patterns)]
X_raw = gk_merged.drop(columns=cols_to_drop, errors='ignore')

# Categoricals to keep
cat_cols = [c for c in ['team', 'league', 'foot', 'agent_name'] if c in gk_merged.columns]
for c in cat_cols:
    X_raw[c] = gk_merged[c].fillna('Unknown').astype(str)

# Numericals to keep
num_cols = [c for c in X_raw.select_dtypes(include=[np.number]).columns if c not in cat_cols]
X = X_raw[cat_cols + num_cols].copy()

# Fill missing keeper-specific metrics
for col in num_cols:
    if any(k in col.lower() for k in ['%', 'save', 'cs', 'clean', 'psxg']):
        X[col] = X[col].fillna(0)
    else:
        med = X[col].median()
        X[col] = X[col].fillna(med if pd.notna(med) else 0)

# ==============================================================================
# 5. PREPROCESSING, TRAINING & SERIALIZATION
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

X_train_encoded = pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())
X_test_encoded = pd.DataFrame(preprocessor.transform(X_test), columns=preprocessor.get_feature_names_out())

# Train XGBoost regressor
xgb_gk = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    tree_method='hist',
    random_state=42
)
xgb_gk.fit(X_train_encoded, y_train)

# Evaluation
y_pred = xgb_gk.predict(X_test_encoded)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

print(f"\n==========================================")
print(f"Goalkeeper XGBoost Performance:")
print(f"R² Score: {r2:.4f}")
print(f"RMSE:     €{rmse:,.2f}")
print(f"==========================================")

# Serialize to models/ directory
model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_gk, model_dir / 'gk_xgboost_model.joblib')
joblib.dump(preprocessor, model_dir / 'gk_preprocessor.joblib')
joblib.dump(cat_cols + num_cols, model_dir / 'gk_feature_order.joblib')

print("Goalkeeper model and preprocessors successfully exported to 'models/'!")

Fuzzy matching 4587 Transfermarkt Goalkeepers...
Successfully matched and merged 1652 goalkeeper records.

Goalkeeper XGBoost Performance:
R² Score: 0.8798
RMSE:     €4,332,435.51
Goalkeeper model and preprocessors successfully exported to 'models/'!


In [6]:
# Save the merged goalkeeper dataset with names for Mode A exploration
gk_merged.to_csv('data/final_gk_data.csv', index=False)
print("Saved data/final_gk_data.csv with goalkeeper names!")

Saved data/final_gk_data.csv with goalkeeper names!
